# 01_clean: turn raw survey codes into an analysis-ready dataset
Input: data/raw/ · Output: data/processed/clean.csv

## 1. Load

In [1]:
import pandas as pd

# -- Load raw data --
# survey microdata: one row per respondent
df = pd.read_csv("../data/raw/polling_on_ai_usage_jul_2026_microdata.csv").copy()

# Epoch's published summary tables: used to check our cleaning
summary = pd.read_csv("../data/raw/polling_on_ai_usage_jul_2026.csv")

print("Microdata:", df.shape)
print("Summary table:", summary.shape)

Microdata: (1103, 333)
Summary table: (3008, 13)


## 2. Demographics

In [2]:
# -- Split Age Groups so it matches Epoch summary table --
df["age_group"] = pd.cut(
    df["ppage"],
    bins=[17, 29, 44, 59, 120],
    labels=["18-29", "30-44", "45-59", "60+"]
)

# counts per group, in order
df["age_group"].value_counts().sort_index()

age_group
18-29    232
30-44    382
45-59    312
60+      177
Name: count, dtype: int64

In [3]:
# Check age groups against Epoch
epoch_age = summary.loc[
    (summary["demographic"] == "age_group") & (summary["question"] == "q2_services"),
    ["demographic_value", "n_base"]
].drop_duplicates() #remove repeats

epoch_age

,demographic_value,n_base
295,18–29,232
304,30–44,382
313,45–59,312
322,60+,177


✅ Age groups match Epoch's n_base exactly.

In [4]:
# -- Education: give the codes readable levels --
edu_labels = {1: "Less than HS", 2: "High school", 3: "Some college", 4: "Bachelor+"}
df["education"] = df["ppeducat"].map(edu_labels)

# college(yes) is considered bachelors degree or higher
df["college"] = df["ppeducat"] == 4

# our counts per education label
print(df["education"].value_counts())

# Epoch count per education label
summary.loc[
    (summary["demographic"] == "education") & (summary["question"] == "q2_services"),
    ["demographic_value", "n_base"]
].drop_duplicates()

education
Bachelor+       470
High school     284
Some college    281
Less than HS     68
Name: count, dtype: int64


,demographic_value,n_base
421,Less than HS,68
430,High school,284
439,Some college,281
448,Bachelor+,470


In [5]:
# -- Income -- 
# Version 1: Epoch's 5 groups --> tables, tool, validation
income_labels = {1: "Under $25K", 2: "Under $25K", 3: "$25K–$50K", 4: "$50K–$75K",
                 5: "$75K–$100K", 6: "$100K+", 7: "$100K+"}
df["income"] = df["ppinc7"].map(income_labels)

# Version 2: same but the top group is split since it holds 57% of sample --> used for regression!
income_reg_labels = {1: "Under $25K", 2: "Under $25K", 3: "$25K–$50K", 4: "$50K–$75K",
                     5: "$75K–$100K", 6: "$100K–$150K", 7: "$150K+"}
df["income_reg"] = df["ppinc7"].map(income_reg_labels)

# -- Race/ethnicity -- 
# Epoch's 4 groups
race_labels = {1: "White, Non-Hispanic", 2: "Black, Non-Hispanic", 3: "Other/2+, Non-Hispanic",
               4: "Hispanic", 5: "Other/2+, Non-Hispanic"}
df["race"] = df["ppethm"].map(race_labels)

# -- Gender -- 
gender_labels = {1: "Male", 2: "Female"}
df["gender"] = df["ppgender"].map(gender_labels)

# Check against Epoch
for ours, theirs in [("income", "income"), ("race", "race_ethnicity"), ("gender", "gender")]:
    print("OURS:", ours)
    print(df[ours].value_counts())
    print("EPOCH:")
    print(summary.loc[
        (summary["demographic"] == theirs) & (summary["question"] == "q2_services"),
        ["demographic_value", "n_base"]
    ].drop_duplicates())
    print("-" * 40)

OURS: income
income
$100K+        634
$50K–$75K     142
$75K–$100K    139
$25K–$50K     109
Under $25K     79
Name: count, dtype: int64
EPOCH:
    demographic_value  n_base
457        Under $25K      79
466         $25K–$50K     109
475         $50K–$75K     142
484        $75K–$100K     139
493            $100K+     634
----------------------------------------
OURS: race
race
White, Non-Hispanic       666
Hispanic                  191
Black, Non-Hispanic       140
Other/2+, Non-Hispanic    106
Name: count, dtype: int64
EPOCH:
          demographic_value  n_base
349     White, Non-Hispanic     666
358     Black, Non-Hispanic     140
367                Hispanic     191
376  Other/2+, Non-Hispanic     106
----------------------------------------
OURS: gender
gender
Male      593
Female    510
Name: count, dtype: int64
EPOCH:
    demographic_value  n_base
331              Male     593
340            Female     510
----------------------------------------


## 3. Job Groups

In [6]:
# -- Job Groups: Q26 code --> 5 groups(overall results + regression) --
job_labels = {
    # Office/Knowledge
    1: "Office/knowledge", 30: "Office/knowledge", 31: "Office/knowledge", 3: "Office/knowledge",
    4: "Office/knowledge", 5: "Office/knowledge", 7: "Office/knowledge", 33: "Office/knowledge",
    22: "Office/knowledge",
    # Sales/service
    35: "Sales/service", 16: "Sales/service", 17: "Sales/service", 18: "Sales/service", 15: "Sales/service",
    # Manual/trades
    23: "Manual/trades", 24: "Manual/trades", 25: "Manual/trades", 26: "Manual/trades", 27: "Manual/trades",
    # Healthcare
    34: "Healthcare", 13: "Healthcare", 14: "Healthcare",
    # Education & social services
    32: "Education & social services", 6: "Education & social services",
}

# codes not in dictionary(29 other, 28 Armed forces, -1 refused) --> NaN
df["job_group"] = df["Q26"].map(job_labels)

# -- 3 Broad groups built from the 5 groups (age x job tables + tool) --
broad_labels = {
    "Office/knowledge": "Office/knowledge",
    "Healthcare": "Care & education",
    "Education & social services": "Care & education",
    "Sales/service": "Frontline",
    "Manual/trades": "Frontline",
}
df["job_broad"] = df["job_group"].map(broad_labels)

# Check
print(df["job_group"].value_counts(dropna=False))
pd.crosstab(df["job_broad"], df["age_group"])

job_group
Office/knowledge               433
Sales/service                  220
Manual/trades                  134
NaN                            120
Education & social services    104
Healthcare                      92
Name: count, dtype: int64


age_group,18-29,30-44,45-59,60+
job_broad,,,,
Care & education,43,68,57,28
Frontline,103,103,92,56
Office/knowledge,63,172,130,68


## 4. Outcome Variables
ai_any = used any AI service in past week
ai_work = used AI at least partly for work (Q5 codes 1-4)

In [7]:
# -- Col lists: Q2_1 ~ Q2_9 = 9 AI services (9 = 'Other'); Q5 = how each was used --
q2_cols = [f"Q2_{i}" for i in range(1, 10)]
q5_cols = [f"Q5_{i}" for i in range(1, 10)]

df["ai_any"] = (df[q2_cols] == 1).any(axis=1).astype(float)

df["ai_work"] = df[q5_cols].isin([1,2,3,4]).any(axis=1).astype(float)

refused_q2 = (df[q2_cols + ["Q2_10"]] == -1).all(axis=1)
df.loc[refused_q2, ["ai_any", "ai_work"]] = None

# Checks
print(df["ai_any"].value_counts(dropna=False))
print(df["ai_work"].value_counts(dropna=False))
pd.crosstab(df["ai_any"], df["ai_work"])

ai_any
1.0    605
0.0    497
NaN      1
Name: count, dtype: int64
ai_work
0.0    633
1.0    469
NaN      1
Name: count, dtype: int64


ai_work,0.0,1.0
ai_any,,
0.0,497,0
1.0,136,469


## 5. Validation: weighted % vs Epoch's publichsed numbers
weighted_pct() = weighted % of 1s in a 0/1 column --> must match Epoch's published pct within 1 percentage point

In [8]:
import sys
sys.path.append("..")
from src.helpers import weighted_pct

# Validation
print("All:", round(100 - weighted_pct(df, "ai_any"), 2))
for group in ["18-29", "30-44", "45-59", "60+"]:
    subgroup = df[df["age_group"] == group]
    print(group, round(100 - weighted_pct(subgroup, "ai_any"), 2))

# Epoch pubclished numbers
summary.loc[
    (summary["question"] == "q2_services") & (summary["response"] == "None (no AI use)")
    & (summary["demographic"].isin(["full_sample", "age_group"])),
    ["demographic_value", "pct", "ci_lo", "ci_hi"]
]

All: 44.64
18-29 41.62
30-44 39.45
45-59 47.72
60+ 56.58


,demographic_value,pct,ci_lo,ci_hi
294,All,44.61,42.14,47.10
303,18–29,41.62,36.39,47.04
312,30–44,39.36,35.32,43.55
321,45–59,47.72,43.09,52.38
330,60+,56.58,50.38,62.58


✅ Matches Epoch within 0.1 percentage points (overall and all 4 age groups).
Tiny differences come from 1 respondent who refused Q2: Epoch kept them in the base; we excluded them.

## 6. Task level time (for Q2)

In [9]:
# --- Task names, in the survey's order ---
task_names = {1: "Maintaining records", 2: "Developing policies/processes", 3: "Reading documents",
              4: "Determining values/prices", 5: "Analyzing data", 6: "Designing systems/apps",
              7: "Preparing documentation", 8: "Explaining info to clients",
              9: "Advising on business", 10: "Assessing students"}

# person level cols to carry along to every task row
keep = ["CaseID", "weight", "age_group", "college", "education", "job_group", "job_broad",
        "gender", "income_reg", "race"]

# build one small table per task, then stack
pieces = []
for k in range(1, 11):
    piece = df[keep].copy()         #person info on everyone
    piece["task"] = task_names[k]   #which task this piece is about
    piece["before"] = df[f"Q13_{k}"] #hours range BEFORE AI for task 'k'
    piece["now"] = df[f"Q11_{k}"]   # hours range NOW for task k
    pieces.append(piece)

tasks = pd.concat(pieces) 
tasks = tasks[tasks["now"].notna()]

# Checks
print("Task reports:", tasks.shape[0], "| People:", tasks["CaseID"].nunique())
tasks.head()

Task reports: 798 | People: 317


,CaseID,weight,age_group,college,education,job_group,job_broad,gender,income_reg,race,task,before,now
24,241,0.9479,30-44,True,Bachelor+,Office/knowledge,Office/knowledge,Male,$150K+,"White, Non-Hispanic",Maintaining records,3.0,1.0
56,1053,0.8084,60+,True,Bachelor+,Education & social services,Care & education,Female,$75K–$100K,"White, Non-Hispanic",Maintaining records,3.0,2.0
95,408,0.9479,30-44,True,Bachelor+,Office/knowledge,Office/knowledge,Male,$150K+,"White, Non-Hispanic",Maintaining records,2.0,2.0
97,573,1.0117,30-44,False,Some college,Office/knowledge,Office/knowledge,Male,$100K–$150K,"Other/2+, Non-Hispanic",Maintaining records,1.0,1.0
201,712,0.8701,30-44,True,Bachelor+,Office/knowledge,Office/knowledge,Male,$100K–$150K,"White, Non-Hispanic",Maintaining records,4.0,3.0


In [10]:
# -- Label each task report -- 
def classify_change(row):
    """
    Label one task report by comparing hours range before and now 
    Codes 1-5 = hour ranges; 6 = not sure; 7 = didn't do this task before AI; -1 = refused
    Returns: "New task", "Not sure/refused", "Less time", "Same", or "More time"
    """

    if row["before"] == 7:
        return "New task"
    if row["before"] in [6, -1] or row["now"] in [6, -1]:
        return "Not sure/refused"
    if row["now"] < row["before"]:
        return "Less time"
    if row["now"] == row["before"]:
        return "Same"
    return "More time"

tasks["change"] = tasks.apply(classify_change, axis="columns")

# Check
tasks["change"].value_counts()

change
Same                322
Less time           281
More time           117
Not sure/refused     57
New task             21
Name: count, dtype: int64

### 6b. Person-level: net time saver
Net time-saver = more tasks got faster than slower(comparable tasks only)

people with no comparable tasks = missing, not "no"

In [11]:
# 1. Keep only comparable task reports
comparable = tasks[tasks["change"].isin(["Less time", "Same", "More time"])].copy()

# 0/1 flags so we can count by adding
comparable["is_less"] = (comparable["change"] == "Less time").astype(int)
comparable["is_more"] = (comparable["change"] == "More time").astype(int)

# 2. Per person: how many tasks got faster vs slower
per_person = comparable.groupby("CaseID")[["is_less", "is_more"]].sum()

# 3. Net time saver = more faster tasks than slower tasks
per_person["net_saver"] = (per_person["is_less"] > per_person["is_more"]).astype(float)

# Attach to df by CaseID; people with no comparable tasks get NaN
df["net_saver"] = df["CaseID"].map(per_person["net_saver"])

# --- 3 way version: separates "no change" from "net slower" ---
per_person["net_status"] = "No change"
per_person.loc[per_person["is_less"] > per_person["is_more"], "net_status"] = "Net saver"
per_person.loc[per_person["is_less"] < per_person["is_more"], "net_status"] = "Net slower"

df["net_status"] = df["CaseID"].map(per_person["net_status"])

# Check
print(per_person["net_status"].value_counts())
print(pd.crosstab(per_person["net_status"], per_person["net_saver"]))


# Checks
print("People with at least 1 comparable task:", len(per_person))
print("People with only new/not sure tasks(excluded):", tasks["CaseID"].nunique() - len(per_person))
per_person.head()

net_status
No change     117
Net saver     114
Net slower     53
Name: count, dtype: int64
net_saver   0.0  1.0
net_status          
Net saver     0  114
Net slower   53    0
No change   117    0
People with at least 1 comparable task: 284
People with only new/not sure tasks(excluded): 33


,is_less,is_more,net_saver,net_status
CaseID,,,,
6,0,0,0.0,No change
9,0,0,0.0,No change
15,0,0,0.0,No change
17,0,0,0.0,No change
18,3,0,1.0,Net saver


## 7. Save

In [12]:
# -- Save the cleaned datasets for 02_analysis --

df.to_csv("../data/processed/clean.csv", index=False)
tasks.to_csv("../data/processed/tasks.csv", index=False)

print("clean.csv:", df.shape)
print("tasks.csv:", tasks.shape)

clean.csv: (1103, 346)
tasks.csv: (798, 14)
